In [57]:
import scipy as sp
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
from pathlib import Path  
import scanpy as sc
import os 
from model_evaluation import *
import warnings
warnings.filterwarnings("ignore")

sc.settings.set_figure_params(figsize=(3, 3))

## Utils functions

In [58]:
def compute_precision_and_recall_auc(predicted, true):
    # Compute precision and recall curve and AUC
    precision, recall, _ = precision_recall_curve(true, predicted)
    pr_auc = auc(recall, precision)
    return pr_auc

def cluster_metrics(abs_ratios, ratios, is_abundant, leiden_labels):
    # Non-abundant clusters should always be lower 
    non_abundant_over_abundant = abs_ratios[~is_abundant].mean() / abs_ratios[is_abundant].mean()
    # Compute fraction correct sign
    ratios_abundant = ratios[is_abundant]  # Only clusters 1 and 2 
    leiden_abundant = np.array(leiden_labels)[is_abundant]

    leiden_abundant_unique = np.unique(leiden_abundant)
    assert "0" not in leiden_abundant_unique
    assert "3" not in leiden_abundant_unique
    
    ratios_abundant[leiden_abundant==2] = ratios_abundant[leiden_abundant==2] * (-1)
    correct_sign_prop = (ratios_abundant >= 0).sum() / len(ratios_abundant)
    
    # Kolmogorov-Smirnov test 
    ks_D, _ = sp.stats.ks_2samp(
        abs_ratios[is_abundant],
        abs_ratios[~is_abundant],
        alternative="two-sided",
        mode="asymp"
    )
    w_distance = sp.stats.wasserstein_distance(abs_ratios[is_abundant], abs_ratios[~is_abundant])
    return non_abundant_over_abundant, correct_sign_prop, ks_D, w_distance
    
def compute_evaluation_metrics(result_path, subsampling_rates, loga_ratio_key):
    # Collect resultsper oversampling (only one per run)
    results_per_oversamp = {
        "p_major_cluster": [], 
        "p_minor_cluster": [],
        "auc_score": [], 
        "non_abundant_over_abundant": [],
        "correct_sign_prop": [],
        }
    
    results_per_run = {
        "metric": [], 
        "value": [],
        }

    # Compute the ratios and abundances per cluster
    per_cluster_ratios = []
    per_cluster_log_odds = []
        
    for r_oversamp in subsampling_rates:
        # Compute the true per-cluster ratio 
        p = 0.5 - r_oversamp  # Always minor cluster prop
        one_minus_p = 1 - p  # Always major cluster prop
        csv_generated = pd.read_csv(result_path / f"oversamp_{r_oversamp}.csv")
            
        # Collect abundance labels 
        is_abundant = np.logical_or(csv_generated.leiden==1, csv_generated.leiden==2)
        abs_ratio = np.abs(csv_generated[loga_ratio_key]).values
        ratio = csv_generated[loga_ratio_key].values

        # Compute AUC
        auc = compute_precision_and_recall_auc(abs_ratio, is_abundant)
        results_per_oversamp["p_major_cluster"].append(one_minus_p)
        results_per_oversamp["p_minor_cluster"].append(p)
        results_per_oversamp["auc_score"].append(auc)
            
        for cl in [0, 1, 2, 3]:
            per_cluster_ratios.append(np.mean(ratio[csv_generated["leiden"]==cl]))

            numerator = (csv_generated.loc[csv_generated["leiden"]==cl, "treatment"]==1).sum()
            denominator = (csv_generated.loc[csv_generated["leiden"]==cl, "treatment"]==0).sum() 

            abs_log_odds = np.log((numerator + 1e-9) / (denominator + 1e-9))
            per_cluster_log_odds.append(abs_log_odds)
            
        # Compute cluster metrics
        abundant_over_non_abundant, correct_sign_prop, _, _ = cluster_metrics(abs_ratio, ratio, is_abundant, csv_generated["leiden"])
        results_per_oversamp["non_abundant_over_abundant"].append(abundant_over_non_abundant)
        results_per_oversamp["correct_sign_prop"].append(correct_sign_prop)
        
    results_per_run["metric"].append("spearman_log_odds")
    results_per_run["value"].append(sp.stats.spearmanr(per_cluster_ratios, per_cluster_log_odds)[0])
    results_per_run["metric"].append("pearson_log_odds")
    results_per_run["value"].append(sp.stats.pearsonr(per_cluster_ratios, per_cluster_log_odds)[0])

    # After computing the metrics per run and abundance, compute metrics per run only. 
    results_per_oversamp_df = pd.DataFrame(results_per_oversamp)
    # Take df at run i 
    for metric in results_per_oversamp_df.columns:
        name = f"corr_{metric}_p"
        results_per_run["metric"].append(name)
        results_per_run["value"].append(sp.stats.pearsonr(results_per_oversamp_df[metric],
                                             results_per_oversamp_df["p_major_cluster"])[0])

        name = f"mean_{metric}"
        results_per_oversamp_df_over_03 = results_per_oversamp_df[results_per_oversamp_df.p_major_cluster>0.8]
        results_per_run["metric"].append(name)
        results_per_run["value"].append(results_per_oversamp_df_over_03[metric].mean())
    
    # Results per data frame 
    results_per_run_df = pd.DataFrame(results_per_run)    
    return results_per_oversamp_df, results_per_run_df

Initialize subsampling rates

In [59]:
subsampling_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.45, 0.50]

Initialize path sweeps 

In [60]:
path_to_sweeps = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/scRatio_sweep")

In [61]:
results = {
    "dimensions": [], 
    "batch_size": [],
    "scheduler_type": [], 
    "sigma": [], 
    "solver": []}


for file in os.listdir(path_to_sweeps):
    # Split file 
    file_split = file.split("_")
    for solver in ["log_ratios_dopri", "log_ratios_euler_100", "log_ratios_euler_200"]:
        results["solver"].append(solver)
        results_per_oversamp_df, results_per_run_df = compute_evaluation_metrics(path_to_sweeps / file,
                                                                                 subsampling_rates, 
                                                                                 solver)
        for key in np.unique(results_per_run_df.metric): 
            if key not in results:
                results[key] = []
            results[key].append(results_per_run_df.loc[results_per_run_df.metric==key]["value"].values[0])
            
    for _ in range(3):
        results["dimensions"].append(file_split[1])
        results["batch_size"].append(file_split[4])
        results["scheduler_type"].append(file_split[7])
        results["sigma"].append(file_split[-1])

In [62]:
results_df = pd.DataFrame(results)

In [63]:
results_df.sort_values(by=["mean_auc_score"], ascending=False)

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
54,10,1024,sigmamin,0.1,log_ratios_dopri,0.939599,0.662755,-0.938396,1.0,-1.0,0.958743,0.986239,0.132426,0.95,0.05,0.933110,0.884531
57,10,1024,stochastic,0.01,log_ratios_dopri,0.904941,0.712695,-0.923284,1.0,-1.0,0.957606,0.986840,0.127402,0.95,0.05,0.921639,0.862537
56,10,1024,sigmamin,0.1,log_ratios_euler_200,0.943798,0.670765,-0.941284,1.0,-1.0,0.957557,0.985956,0.133698,0.95,0.05,0.933545,0.884531
0,10,1024,sigmamin,0.01,log_ratios_dopri,0.899868,0.699751,-0.894517,1.0,-1.0,0.956345,0.985570,0.128625,0.95,0.05,0.932837,0.884897
36,10,1024,stochastic,0.1,log_ratios_dopri,0.876108,0.690504,-0.855146,1.0,-1.0,0.956102,0.985040,0.131763,0.95,0.05,0.926615,0.855938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,20,256,stochastic,0.1,log_ratios_euler_200,0.969178,0.827886,-0.890570,1.0,-1.0,0.886489,0.973414,0.207185,0.95,0.05,0.905044,0.832845
10,20,512,sigmamin,0.1,log_ratios_euler_100,0.991397,0.899519,-0.969156,1.0,-1.0,0.883613,0.966020,0.209496,0.95,0.05,0.940625,0.829179
88,20,1024,stochastic,0.1,log_ratios_euler_100,0.959153,0.825071,-0.583948,1.0,-1.0,0.883565,0.972955,0.260067,0.95,0.05,0.895150,0.840909
82,20,256,sigmamin,0.01,log_ratios_euler_100,0.982268,0.891016,-0.968203,1.0,-1.0,0.881969,0.971313,0.217762,0.95,0.05,0.918834,0.807185


In [64]:
results_df_sigma_min = results_df[results_df.scheduler_type=="sigmamin"]

In [65]:
results_df_stochastic = results_df[results_df.scheduler_type=="stochastic"]

In [66]:
results_df_deterministic = results_df[results_df.scheduler_type=="deterministic"]

In [67]:
results_df_sigma_min.sort_values(by=["mean_auc_score"], ascending=False)

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
54,10,1024,sigmamin,0.1,log_ratios_dopri,0.939599,0.662755,-0.938396,1.0,-1.0,0.958743,0.986239,0.132426,0.95,0.05,0.933110,0.884531
56,10,1024,sigmamin,0.1,log_ratios_euler_200,0.943798,0.670765,-0.941284,1.0,-1.0,0.957557,0.985956,0.133698,0.95,0.05,0.933545,0.884531
0,10,1024,sigmamin,0.01,log_ratios_dopri,0.899868,0.699751,-0.894517,1.0,-1.0,0.956345,0.985570,0.128625,0.95,0.05,0.932837,0.884897
30,10,256,sigmamin,0.1,log_ratios_dopri,0.929346,0.712385,-0.926721,1.0,-1.0,0.955003,0.983002,0.131674,0.95,0.05,0.910642,0.869135
2,10,1024,sigmamin,0.01,log_ratios_euler_200,0.905121,0.702652,-0.898922,1.0,-1.0,0.954770,0.985501,0.130298,0.95,0.05,0.933179,0.875733
55,10,1024,sigmamin,0.1,log_ratios_euler_100,0.949894,0.687730,-0.946801,1.0,-1.0,0.954744,0.984933,0.136741,0.95,0.05,0.933810,0.884897
32,10,256,sigmamin,0.1,log_ratios_euler_200,0.931467,0.712669,-0.926386,1.0,-1.0,0.954353,0.982542,0.132878,0.95,0.05,0.911020,0.869135
31,10,256,sigmamin,0.1,log_ratios_euler_100,0.938575,0.729648,-0.932998,1.0,-1.0,0.952370,0.982120,0.135732,0.95,0.05,0.911659,0.868768
1,10,1024,sigmamin,0.01,log_ratios_euler_100,0.917396,0.720042,-0.912350,1.0,-1.0,0.952263,0.983421,0.133040,0.95,0.05,0.933938,0.875733
48,10,256,sigmamin,0.01,log_ratios_dopri,0.911002,0.686331,-0.914822,1.0,-1.0,0.950895,0.983654,0.138981,0.95,0.05,0.909005,0.902493


In [68]:
results_df_stochastic.sort_values(by=["mean_auc_score"], ascending=False).head()

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
57,10,1024,stochastic,0.01,log_ratios_dopri,0.904941,0.712695,-0.923284,1.0,-1.0,0.957606,0.986840,0.127402,0.95,0.05,0.921639,0.862537
36,10,1024,stochastic,0.1,log_ratios_dopri,0.876108,0.690504,-0.855146,1.0,-1.0,0.956102,0.985040,0.131763,0.95,0.05,0.926615,0.855938
59,10,1024,stochastic,0.01,log_ratios_euler_200,0.909262,0.714325,-0.925457,1.0,-1.0,0.955962,0.986381,0.129005,0.95,0.05,0.922005,0.859971
38,10,1024,stochastic,0.1,log_ratios_euler_200,0.882531,0.698001,-0.860488,1.0,-1.0,0.954403,0.983965,0.133989,0.95,0.05,0.927466,0.857405
58,10,1024,stochastic,0.01,log_ratios_euler_100,0.917729,0.724647,-0.932096,1.0,-1.0,0.952979,0.985075,0.132103,0.95,0.05,0.922489,0.862537


In [69]:
results_df_deterministic.sort_values(by=["mean_auc_score"], ascending=False).head()

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
84,10,512,deterministic,0,log_ratios_dopri,0.934200,0.697242,-0.931840,1.0,-1.0,0.954244,0.984880,0.136330,0.95,0.05,0.928486,0.845308
75,10,1024,deterministic,0,log_ratios_dopri,0.897445,0.704520,-0.922071,1.0,-1.0,0.953368,0.983966,0.139672,0.95,0.05,0.929971,0.850440
86,10,512,deterministic,0,log_ratios_euler_200,0.936867,0.694617,-0.933471,1.0,-1.0,0.953251,0.984476,0.137543,0.95,0.05,0.929021,0.845308
77,10,1024,deterministic,0,log_ratios_euler_200,0.899867,0.706127,-0.922942,1.0,-1.0,0.952143,0.982768,0.140957,0.95,0.05,0.930431,0.849707
39,10,256,deterministic,0,log_ratios_dopri,0.887591,0.698249,-0.882643,1.0,-1.0,0.952128,0.983963,0.141926,0.95,0.05,0.925114,0.838343


## Check per batch size - deterministic

In [73]:
results_df_deterministic[results_df_deterministic.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
39,10,256,deterministic,0,log_ratios_dopri,0.887591,0.698249,-0.882643,1.0,-1.0,0.952128,0.983963,0.141926,0.95,0.05,0.925114,0.838343
40,10,256,deterministic,0,log_ratios_euler_100,0.902656,0.713339,-0.896007,1.0,-1.0,0.949552,0.982625,0.144740,0.95,0.05,0.926293,0.838343
41,10,256,deterministic,0,log_ratios_euler_200,0.889038,0.699022,-0.882370,1.0,-1.0,0.951092,0.983681,0.142673,0.95,0.05,0.925912,0.838343
60,20,256,deterministic,0,log_ratios_dopri,0.978926,0.821166,-0.935249,1.0,-1.0,0.911421,0.972579,0.172187,0.95,0.05,0.933936,0.802786
61,20,256,deterministic,0,log_ratios_euler_100,0.989027,0.845764,-0.945519,1.0,-1.0,0.903346,0.969404,0.182468,0.95,0.05,0.934329,0.805352
62,20,256,deterministic,0,log_ratios_euler_200,0.984219,0.830964,-0.942218,1.0,-1.0,0.909178,0.971116,0.176063,0.95,0.05,0.934219,0.795088


In [74]:
results_df_deterministic[results_df_deterministic.batch_size=="512"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
84,10,512,deterministic,0,log_ratios_dopri,0.934200,0.697242,-0.931840,1.0,-1.0,0.954244,0.984880,0.136330,0.95,0.05,0.928486,0.845308
85,10,512,deterministic,0,log_ratios_euler_100,0.943091,0.709818,-0.939627,1.0,-1.0,0.950088,0.983420,0.140313,0.95,0.05,0.929553,0.849707
86,10,512,deterministic,0,log_ratios_euler_200,0.936867,0.694617,-0.933471,1.0,-1.0,0.953251,0.984476,0.137543,0.95,0.05,0.929021,0.845308
18,20,512,deterministic,0,log_ratios_dopri,0.993779,0.882244,-0.977447,1.0,-1.0,0.913661,0.966938,0.157338,0.95,0.05,0.937955,0.873167
19,20,512,deterministic,0,log_ratios_euler_100,0.995665,0.905854,-0.982384,1.0,-1.0,0.907147,0.964729,0.163835,0.95,0.05,0.937101,0.887830
20,20,512,deterministic,0,log_ratios_euler_200,0.994247,0.893494,-0.977654,1.0,-1.0,0.911743,0.966185,0.159063,0.95,0.05,0.937359,0.872067


In [75]:
results_df_deterministic[results_df_deterministic.batch_size=="1024"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
75,10,1024,deterministic,0,log_ratios_dopri,0.897445,0.704520,-0.922071,1.0,-1.0,0.953368,0.983966,0.139672,0.95,0.05,0.929971,0.850440
76,10,1024,deterministic,0,log_ratios_euler_100,0.907922,0.721022,-0.926872,1.0,-1.0,0.949112,0.982046,0.143618,0.95,0.05,0.931091,0.843842
77,10,1024,deterministic,0,log_ratios_euler_200,0.899867,0.706127,-0.922942,1.0,-1.0,0.952143,0.982768,0.140957,0.95,0.05,0.930431,0.849707
21,20,1024,deterministic,0,log_ratios_dopri,0.975866,0.833004,-0.951530,1.0,-1.0,0.910640,0.973808,0.182775,0.95,0.05,0.940691,0.842009
22,20,1024,deterministic,0,log_ratios_euler_100,0.986281,0.875204,-0.969410,1.0,-1.0,0.895333,0.970135,0.197646,0.95,0.05,0.941000,0.822947
23,20,1024,deterministic,0,log_ratios_euler_200,0.980655,0.847290,-0.959244,1.0,-1.0,0.904172,0.971944,0.189346,0.95,0.05,0.941087,0.836877


The batch size has effect on: 
* corr auc score - better for 512 
* corr abundant over non abundant better for 20 and for 512
* Everything is perfectly reproducible 

## Check per batch size - stochastic

In [77]:
results_df_stochastic[results_df_stochastic.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
63,10,256,stochastic,0.1,log_ratios_dopri,0.946881,0.712128,-0.935947,1.0,-1.0,0.942988,0.981934,0.151776,0.95,0.05,0.893384,0.829912
64,10,256,stochastic,0.1,log_ratios_euler_100,0.954750,0.724642,-0.942512,1.0,-1.0,0.940646,0.980013,0.155356,0.95,0.05,0.894628,0.829912
65,10,256,stochastic,0.1,log_ratios_euler_200,0.948950,0.715017,-0.935849,1.0,-1.0,0.942054,0.981052,0.153215,0.95,0.05,0.893937,0.829912
69,10,256,stochastic,0.01,log_ratios_dopri,0.874472,0.724208,-0.876338,1.0,-1.0,0.945640,0.983898,0.148188,0.95,0.05,0.932491,0.841642
70,10,256,stochastic,0.01,log_ratios_euler_100,0.884296,0.731534,-0.882363,1.0,-1.0,0.942030,0.982469,0.152359,0.95,0.05,0.933465,0.841642
71,10,256,stochastic,0.01,log_ratios_euler_200,0.874624,0.721831,-0.875809,1.0,-1.0,0.944521,0.983721,0.149255,0.95,0.05,0.933132,0.842375
27,20,256,stochastic,0.01,log_ratios_dopri,0.995330,0.857582,-0.973888,1.0,-1.0,0.910753,0.978963,0.188036,0.95,0.05,0.919025,0.840543
28,20,256,stochastic,0.01,log_ratios_euler_100,0.994366,0.883428,-0.978658,1.0,-1.0,0.897879,0.975013,0.200150,0.95,0.05,0.918767,0.825880
29,20,256,stochastic,0.01,log_ratios_euler_200,0.994837,0.870514,-0.977323,1.0,-1.0,0.905409,0.977025,0.193763,0.95,0.05,0.919182,0.833944
33,20,256,stochastic,0.1,log_ratios_dopri,0.962263,0.817493,-0.885002,1.0,-1.0,0.891928,0.975528,0.201129,0.95,0.05,0.904871,0.823680


In [78]:
results_df_stochastic[results_df_stochastic.batch_size=="512"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
45,10,512,stochastic,0.1,log_ratios_dopri,0.939219,0.721525,-0.939137,1.0,-1.0,0.947635,0.981066,0.132765,0.95,0.05,0.925389,0.854839
46,10,512,stochastic,0.1,log_ratios_euler_100,0.948917,0.742395,-0.948575,1.0,-1.0,0.944194,0.979319,0.138045,0.95,0.05,0.926508,0.850073
47,10,512,stochastic,0.1,log_ratios_euler_200,0.943427,0.727261,-0.940613,1.0,-1.0,0.946241,0.980361,0.134934,0.95,0.05,0.926038,0.853372
51,10,512,stochastic,0.01,log_ratios_dopri,0.880939,0.669154,-0.886517,1.0,-1.0,0.925141,0.981542,0.171314,0.95,0.05,0.921030,0.860337
52,10,512,stochastic,0.01,log_ratios_euler_100,0.889829,0.682356,-0.897285,1.0,-1.0,0.918248,0.980183,0.176606,0.95,0.05,0.921900,0.860337
53,10,512,stochastic,0.01,log_ratios_euler_200,0.881989,0.669612,-0.887656,1.0,-1.0,0.922401,0.980908,0.173355,0.95,0.05,0.921425,0.860337
6,20,512,stochastic,0.1,log_ratios_dopri,0.968054,0.858423,-0.917576,1.0,-1.0,0.917597,0.979357,0.173624,0.95,0.05,0.918173,0.857771
7,20,512,stochastic,0.1,log_ratios_euler_100,0.984039,0.892103,-0.973275,1.0,-1.0,0.898799,0.974541,0.194976,0.95,0.05,0.920333,0.854839
8,20,512,stochastic,0.1,log_ratios_euler_200,0.976225,0.872625,-0.957332,1.0,-1.0,0.909589,0.977109,0.182815,0.95,0.05,0.919318,0.846774
15,20,512,stochastic,0.01,log_ratios_dopri,0.980989,0.844474,-0.955010,1.0,-1.0,0.934336,0.977026,0.148504,0.95,0.05,0.951509,0.869501


In [79]:
results_df_stochastic[results_df_stochastic.batch_size=="1024"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
36,10,1024,stochastic,0.1,log_ratios_dopri,0.876108,0.690504,-0.855146,1.0,-1.0,0.956102,0.985040,0.131763,0.95,0.05,0.926615,0.855938
37,10,1024,stochastic,0.1,log_ratios_euler_100,0.895628,0.715225,-0.873426,1.0,-1.0,0.951089,0.982890,0.137537,0.95,0.05,0.928213,0.856672
38,10,1024,stochastic,0.1,log_ratios_euler_200,0.882531,0.698001,-0.860488,1.0,-1.0,0.954403,0.983965,0.133989,0.95,0.05,0.927466,0.857405
57,10,1024,stochastic,0.01,log_ratios_dopri,0.904941,0.712695,-0.923284,1.0,-1.0,0.957606,0.986840,0.127402,0.95,0.05,0.921639,0.862537
58,10,1024,stochastic,0.01,log_ratios_euler_100,0.917729,0.724647,-0.932096,1.0,-1.0,0.952979,0.985075,0.132103,0.95,0.05,0.922489,0.862537
59,10,1024,stochastic,0.01,log_ratios_euler_200,0.909262,0.714325,-0.925457,1.0,-1.0,0.955962,0.986381,0.129005,0.95,0.05,0.922005,0.859971
12,20,1024,stochastic,0.01,log_ratios_dopri,0.985007,0.841227,-0.967163,1.0,-1.0,0.924799,0.974773,0.158832,0.95,0.05,0.947365,0.869135
13,20,1024,stochastic,0.01,log_ratios_euler_100,0.990010,0.876136,-0.978751,1.0,-1.0,0.913849,0.970834,0.172675,0.95,0.05,0.947580,0.856305
14,20,1024,stochastic,0.01,log_ratios_euler_200,0.987710,0.857014,-0.971762,1.0,-1.0,0.920821,0.973590,0.164465,0.95,0.05,0.947334,0.853372
87,20,1024,stochastic,0.1,log_ratios_dopri,0.953467,0.748055,-0.532205,1.0,-1.0,0.890143,0.978467,0.353052,0.95,0.05,0.557283,0.795821


## Check per batch size - sigma min

In [80]:
results_df_sigma_min[results_df_sigma_min.batch_size=="256"].sort_values(by="dimensions")

,dimensions,batch_size,scheduler_type,sigma,solver,corr_auc_score_p,corr_correct_sign_prop_p,corr_non_abundant_over_abundant_p,corr_p_major_cluster_p,corr_p_minor_cluster_p,mean_auc_score,mean_correct_sign_prop,mean_non_abundant_over_abundant,mean_p_major_cluster,mean_p_minor_cluster,pearson_log_odds,spearman_log_odds
30,10,256,sigmamin,0.1,log_ratios_dopri,0.929346,0.712385,-0.926721,1.0,-1.0,0.955003,0.983002,0.131674,0.95,0.05,0.910642,0.869135
31,10,256,sigmamin,0.1,log_ratios_euler_100,0.938575,0.729648,-0.932998,1.0,-1.0,0.952370,0.982120,0.135732,0.95,0.05,0.911659,0.868768
32,10,256,sigmamin,0.1,log_ratios_euler_200,0.931467,0.712669,-0.926386,1.0,-1.0,0.954353,0.982542,0.132878,0.95,0.05,0.911020,0.869135
48,10,256,sigmamin,0.01,log_ratios_dopri,0.911002,0.686331,-0.914822,1.0,-1.0,0.950895,0.983654,0.138981,0.95,0.05,0.909005,0.902493
49,10,256,sigmamin,0.01,log_ratios_euler_100,0.921747,0.700089,-0.923705,1.0,-1.0,0.948559,0.982720,0.142323,0.95,0.05,0.909798,0.902493
50,10,256,sigmamin,0.01,log_ratios_euler_200,0.912370,0.687361,-0.916564,1.0,-1.0,0.950064,0.983318,0.140098,0.95,0.05,0.909333,0.897361
24,20,256,sigmamin,0.1,log_ratios_dopri,0.986400,0.845566,-0.962404,1.0,-1.0,0.910679,0.975406,0.179387,0.95,0.05,0.906913,0.838710
25,20,256,sigmamin,0.1,log_ratios_euler_100,0.988677,0.872326,-0.971515,1.0,-1.0,0.900604,0.972565,0.190837,0.95,0.05,0.906073,0.829912
26,20,256,sigmamin,0.1,log_ratios_euler_200,0.987433,0.858082,-0.966926,1.0,-1.0,0.906964,0.973976,0.184057,0.95,0.05,0.906385,0.828812
81,20,256,sigmamin,0.01,log_ratios_dopri,0.976306,0.870828,-0.962810,1.0,-1.0,0.898203,0.974835,0.203344,0.95,0.05,0.917823,0.814150
